# Scaling classification with BERT (Stage 2, Steps 7–9)

Companion code for the *Organization Studies* paper “Measuring Organizational Constructs in Large-Scale Text: An interpretive framework combining topic modeling and BERT”, implementing Stage 2 of the framework: fine-tuning BERT-base for multi-label classification of employee reviews into 12 constructs plus a residual *Others* category, using the labeled dataset constructed in Step 6.

Correspondence to the paper:

- **Step 7 — BERT text preprocessing (Section 3.7):** BERT-base-uncased tokenization; padding to fixed length, truncation at 512 tokens, attention masks.
- **Step 8 — BERT fine-tuning and evaluation (Section 3.8):** hyperparameter optimization (Optuna), 5-fold multilabel-stratified cross-validation with class-weighted BCE loss and early stopping, per-label classification-threshold tuning, and final training on 100% of the training set.
- **Step 9 — Final model validation on held-out test data (Section 3.9):** single evaluation of the final model on the held-out test set, and inference with the trained model (`predict_labels`).

**Data:** expects `train_data.csv` and `test_data.csv` (columns: id, `pros` review text, one binary indicator per construct). The review texts are not redistributed; users supply their own labeled data (see Data Availability). The label column `Externally generated goals` corresponds to the construct Job-Specific Demands in the paper — the column retains the name of Warr's umbrella category, whose two components (Job-Specific Demands and Work–Life Balance) are treated as separate constructs (see Appendix 3).

**Reproducibility:** random seeds fixed at 42 (train/validation split, CV folds). Transformer fine-tuning on GPU is not bit-for-bit deterministic, so metrics may vary marginally across runs and hardware. The Optuna search was run once; the selected hyperparameters are hard-coded below, so results can be reproduced without re-running the search. Developed and run on a server-grade environment (48 vCPUs; 192 GB RAM; NVIDIA H100 GPU).

## Setup and dependencies

In [ ]:
# Install dependencies (run once per fresh environment)
!pip install --upgrade accelerate transformers
!pip install scikit-learn openpyxl optuna iterative-stratification seaborn

In [ ]:
import os
import torch
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score, precision_recall_fscore_support, precision_score, recall_score, f1_score, accuracy_score
from torch.utils.data import Dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
import torch.nn as nn
import json

start_time = time.time()
torch.cuda.empty_cache()

## Data loading

In [ ]:
# Load dataset
file_path = "/data/train_data.csv"   # user-supplied labeled training data (not redistributed; see Data Availability)
data = pd.read_csv(file_path)
data['pros'] = data['pros'].astype(str).fillna("") # Convert to string and handle missing values

# Extract features and labels
features = data[['pros']].dropna() # Extract text features, removing any rows with missing values
topic_columns = data.columns[2:]  # Select all columns after 'pros'
labels = data[topic_columns].fillna(0).astype(int) # Convert labels to integers, filling missing values with 0
text_data = features['pros'].tolist() #Convert to list for processing

# Display label distribution
label_counts = labels.sum().sort_values(ascending=True)
print("\nLabel distribution:")
print(label_counts)

## Step 7 — BERT text preprocessing

In [ ]:

# Initialize BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Custom Dataset class for text data
class TextDataset(Dataset):
    """Custom dataset class for handling text data with multi-label classification."""
    def __init__(self, texts, labels=None, tokenizer=None, max_token_len=512):
        self.texts = texts
        self.labels = labels.reset_index(drop=True) if isinstance(labels, pd.DataFrame) else labels
        self.tokenizer = tokenizer
        self.max_token_len = max_token_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        inputs = self.tokenizer(
            self.texts[idx],
            add_special_tokens=True,
            max_length=self.max_token_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True
        )

        item = {
            'input_ids': torch.tensor(inputs['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(inputs['attention_mask'], dtype=torch.long)
        }

        if self.labels is not None:
            label_row = self.labels.iloc[idx].values if hasattr(self.labels, 'iloc') else self.labels[idx]
            item['labels'] = torch.tensor(label_row, dtype=torch.float)

        return item

In [ ]:
# Token-length audit (Section 3.7): full labeled dataset (657 = 530 training + 127 test documents)
# WordPiece counts without [CLS]/[SEP]
test_texts    = pd.read_csv("/data/test_data.csv")['pros'].astype(str).fillna("").tolist()
labeled_texts = text_data + test_texts        # reconstruct the labeled dataset from its two partitions
token_lengths = [len(tokenizer.tokenize(t)) for t in labeled_texts]
print(f"Documents: {len(token_lengths)}")                                 # expected: 657
print(f"Average tokens: {np.mean(token_lengths):.1f}")                    # expected: 20.2
print(f"Min / Max tokens: {min(token_lengths)} / {max(token_lengths)}")   # expected: 5 / 373
# All labeled documents fit within BERT's 512-token input limit -> no truncation (Section 3.7)

## Step 8 — BERT fine-tuning and evaluation

In [ ]:
# Calculate class weights for handling imbalanced labels
total_samples = len(labels)
class_weights = {}
for i, label in enumerate(topic_columns):
    # Count positive samples for this label
    pos_count = labels[label].sum()
    # Calculate weight as inverse of frequency
    weight = total_samples / (2 * pos_count) if pos_count > 0 else 1.0
    class_weights[label] = weight

# Print class weights
print("\nClass weights for handling imbalance:")
for label, weight in class_weights.items():
    print(f"{label}: {weight:.4f}")


*Optional — the hyperparameter search below takes several hours. To reproduce the paper's results, skip this cell and run the next one, which hard-codes the selected configuration.*

In [ ]:
# === OPTUNA HYPERPARAMETER OPTIMIZATION ===
import optuna
from transformers import TrainingArguments, Trainer
from sklearn.model_selection import train_test_split

def model_init():
    return BertForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=len(topic_columns),
        problem_type="multi_label_classification"
    )

# Split data for validation
train_texts, val_texts, train_labels, val_labels = train_test_split(
    text_data, labels, test_size=0.2, random_state=42
)

train_dataset = TextDataset(train_texts, train_labels, tokenizer)
val_dataset = TextDataset(val_texts, val_labels, tokenizer)

# Objective function for Optuna
def objective(trial):
    # Suggest hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 5e-5, 2e-4, log=True)
    batch_size = trial.suggest_categorical("batch_size", [2, 4, 8, 16])
    weight_decay = trial.suggest_float("weight_decay", 0.001, 0.05)
    num_train_epochs = trial.suggest_int("epochs", 10, 30)

    # Training arguments
    args = TrainingArguments(
        output_dir="./optuna_trial",
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="no",
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        num_train_epochs=num_train_epochs,
        load_best_model_at_end=False,
        metric_for_best_model=None,
        report_to="none",
        fp16=True
    )

    trainer = Trainer(
        model_init=model_init,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=None
    )

    trainer.train()
    eval_results = trainer.evaluate()
    return eval_results["eval_loss"]

import time

# Start timer
start_time = time.time()

# Run the Optuna study
study = optuna.create_study(direction="minimize",
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1))
study.optimize(objective, n_trials=100)

# End timer
end_time = time.time()

# Calculate elapsed time
elapsed_time = end_time - start_time

# Print in minutes and seconds
minutes, seconds = divmod(elapsed_time, 60)
print(f"Optimization completed in {int(minutes)} minutes and {int(seconds)} seconds")

# Print best trial
print("Best trial:")
print(study.best_trial)

In [ ]:
#best_params = study.best_trial.params

#best_learning_rate = best_params["learning_rate"]
#best_batch_size = best_params["batch_size"]
#best_weight_decay = best_params["weight_decay"]
#best_num_epochs = best_params["epochs"]

#best hyperparameters found with Optuna
best_learning_rate = 9.307187791128257e-05
best_batch_size = 8
best_weight_decay = 0.03510684859073831
best_num_epochs = 20


In [ ]:
print("=== Best Hyperparameters Found ===")
print(f"Learning Rate     : {best_learning_rate}")
print(f"Batch Size        : {best_batch_size}")
print(f"Weight Decay      : {best_weight_decay}")
print(f"Number of Epochs  : {best_num_epochs}")


In [ ]:
#Adjusting more the weights to balance Precision-Recall for rare cases (optional, after seeing precision-recall outcome) # new
adjusted_class_weights = class_weights.copy()
#adjusted_class_weights["Clarity"] *= 0.6 # retain only 0.6 from initial weight
#adjusted_class_weights["Control"] *= 0.6

In [ ]:
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

In [ ]:
# Custom Trainer class with weighted BCE loss for multi-label classification
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # MODIFIED: Use weighted BCE loss to handle class imbalance
        pos_weights = torch.ones(len(topic_columns), device=logits.device)

        # Set weights for each label based on class distribution
        for i, label in enumerate(topic_columns):
            pos_weights[i] = torch.tensor(adjusted_class_weights[label])            # new : added adjusted_ weights from above

        # Use BCEWithLogitsLoss with pos_weight parameter
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=pos_weights)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

# Metric function
def compute_metrics(pred):
    """Compute precision, recall, and F1 score for model evaluation."""
    labels = pred.label_ids
    preds = (pred.predictions > 0).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='micro', zero_division=0)
    return {
        "eval_precision": precision,
        "eval_recall": recall,
        "eval_f1": f1
    }


# Use Optuna's best hyperparameters directly
learning_rates = [best_learning_rate]
batch_sizes = [best_batch_size]
epochs_list = [best_num_epochs]
weight_decays = [best_weight_decay]



# K-Fold setup
k_folds = 5
mskf = MultilabelStratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

## Initialize lists to store results
results = []
training_logs = []
all_fold_thresholds = []  # <--- collect tuned thresholds per fold
per_label_results = []    # <--- per-construct metrics within each fold

# Cross-validation (hyperparameters fixed to Optuna's best configuration)
for fold, (train_idx, val_idx) in enumerate(mskf.split(text_data, labels.values)):
    print(f"\nTraining fold {fold+1}/{k_folds}")

    # Create train and validation datasets for this fold
    train_dataset = TextDataset(
        texts=[text_data[i] for i in train_idx],
        labels=labels.iloc[train_idx],
        tokenizer=tokenizer
    )
    val_dataset = TextDataset(
        texts=[text_data[i] for i in val_idx],
        labels=labels.iloc[val_idx],
        tokenizer=tokenizer
    )

    # Train with the selected hyperparameters (single configuration from Optuna)
    for lr in learning_rates:
        for batch_size in batch_sizes:
            for num_epochs in epochs_list:
                for wd in weight_decays:
                    print(f"Fold {fold+1} | LR: {lr} | BS: {batch_size} | Epochs: {num_epochs} | WD: {wd}")

                    training_args = TrainingArguments(
                        output_dir=f'./bert_fold{fold+1}_lr{lr}_bs{batch_size}_ep{num_epochs}_wd{wd}',
                        num_train_epochs=num_epochs,
                        per_device_train_batch_size=batch_size,
                        per_device_eval_batch_size=batch_size * 2,
                        learning_rate=lr,
                        weight_decay=wd,
                        eval_strategy='epoch',
                        save_strategy='epoch',
                        save_total_limit=2,
                        load_best_model_at_end=True,
                        metric_for_best_model="eval_loss",
                        greater_is_better=False,
                        gradient_accumulation_steps=2,
                        warmup_steps=60,  # <-- NEW LINE
                        fp16=True
                    )

                    model = BertForSequenceClassification.from_pretrained(
                        "bert-base-uncased",
                        num_labels=len(topic_columns),
                        problem_type="multi_label_classification"
                    )

                    trainer = CustomTrainer(
                        model=model,
                        args=training_args,
                        train_dataset=train_dataset,
                        eval_dataset=val_dataset,
                        compute_metrics=compute_metrics,
                        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
                    )

                    trainer.train()
                    trainer.save_model(training_args.output_dir)
                    trainer.save_state()

                    eval_results = trainer.evaluate()
                    predictions = trainer.predict(val_dataset).predictions
                    probas = torch.sigmoid(torch.tensor(predictions)).numpy()
                    true_labels = labels.iloc[val_idx].values

                    # Dynamic threshold tuning function with class balance awareness          --- NEW
                    def tune_thresholds(y_true, y_probs, label_names):
                        best_thresholds = {}
                        for i, label in enumerate(label_names):
                            # Count positive examples for this label
                            positive_count = y_true[:, i].sum()
                            total_count = len(y_true)
                            class_ratio = positive_count / total_count

                            print(f"Label: {label}, Positive ratio: {class_ratio:.4f}")

                            # For rare classes (less than 10%), use lower thresholds
                            if class_ratio < 0.1:
                                # Search in a lower range for rare classes
                                threshold_range = np.arange(0.05, 0.45, 0.05)
                                print(f"  Using lower threshold range for rare class: {threshold_range}")
                            else:
                                # Standard range for common classes
                                threshold_range = np.arange(0.1, 0.70, 0.05)

                            best_f1 = 0
                            best_t = 0.5
                            for t in threshold_range:
                                preds = (y_probs[:, i] > t).astype(int)
                                f1 = f1_score(y_true[:, i], preds, zero_division=0)
                                if f1 > best_f1:
                                    best_f1 = f1
                                    best_t = t

                            print(f"  Best threshold for {label}: {best_t} (F1: {best_f1:.4f})")
                            best_thresholds[label] = best_t
                        return best_thresholds


                    best_thresholds = tune_thresholds(true_labels, probas, labels.columns)

                    all_fold_thresholds.append({
                        "fold": fold + 1,
                        "lr": lr,
                        "batch_size": batch_size,
                        "epochs": num_epochs,
                        "weight_decay": wd,
                        "thresholds": best_thresholds
                    })

                    tuned_preds = (probas >= np.array([best_thresholds[label] for label in labels.columns])).astype(int)

                    results.append({
                        "fold": fold + 1,
                        "learning_rate": lr,
                        "batch_size": batch_size,
                        "epochs": num_epochs,
                        "weight_decay": wd,
                        "eval_loss": eval_results.get("eval_loss"),
                        "eval_precision": precision_score(true_labels, tuned_preds, average='micro', zero_division=0),
                        "eval_recall": recall_score(true_labels, tuned_preds, average='micro', zero_division=0),
                        "eval_f1": f1_score(true_labels, tuned_preds, average='micro', zero_division=0),
                        "eval_accuracy": accuracy_score(true_labels.flatten(), tuned_preds.flatten()),
                        "cohen_kappa": cohen_kappa_score(true_labels.flatten(), tuned_preds.flatten())
                    })

                    # Per-construct metrics within this fold (using the fold's tuned thresholds)
                    for i, label in enumerate(topic_columns):
                        per_label_results.append({
                            "fold": fold + 1,
                            "label": label,
                            "precision":   precision_score(true_labels[:, i], tuned_preds[:, i], zero_division=0),
                            "recall":      recall_score(true_labels[:, i], tuned_preds[:, i], zero_division=0),
                            "f1":          f1_score(true_labels[:, i], tuned_preds[:, i], zero_division=0),
                            "accuracy":    accuracy_score(true_labels[:, i], tuned_preds[:, i]),
                            "cohen_kappa": cohen_kappa_score(true_labels[:, i], tuned_preds[:, i])
                        })

                    training_logs.extend(trainer.state.log_history)

# Save tuned thresholds to file
with open("tuned_thresholds_per_fold.json", "w") as f:
    json.dump(all_fold_thresholds, f, indent=4)

print("\n Saved tuned thresholds to 'tuned_thresholds_per_fold.json'")



# === Average thresholds across folds ===
label_thresholds = {label: [] for label in labels.columns}
for fold_result in all_fold_thresholds:
    for label, threshold in fold_result["thresholds"].items():
        label_thresholds[label].append(threshold)

avg_thresholds = {label: np.mean(thresh_list) for label, thresh_list in label_thresholds.items()}

# Save averaged thresholds to file
with open("optimal_thresholds.json", "w") as f:
    json.dump(avg_thresholds, f, indent=4)

print("\n Saved averaged thresholds to 'optimal_thresholds.json'")




# === Save CV results and summary ===
results_df = pd.DataFrame(results)
results_df.to_excel("kfold_performance_metrics_bert.xlsx", index=False)

metric_cols = ["eval_loss", "eval_precision", "eval_recall", "eval_f1", "eval_accuracy", "cohen_kappa"]
best_row  = results_df.sort_values("eval_loss").iloc[0]
worst_row = results_df.sort_values("eval_loss", ascending=False).iloc[0]
avg_metrics = results_df[metric_cols].mean()

# Per-construct CV metrics, averaged across folds (Table 7, construct level)
per_label_df = pd.DataFrame(per_label_results)
per_label_cv = (per_label_df.groupby("label")[["precision", "recall", "f1", "accuracy", "cohen_kappa"]]
                .mean().reset_index())
per_label_cv.to_excel("per_construct_cv_metrics.xlsx", index=False)
print(per_label_cv)

# Overall summary (Table 7, overall rows)
summary_df = pd.DataFrame({
    "Best Hyperparameters": best_row[["learning_rate", "batch_size", "epochs", "weight_decay"]],
    "Average Metrics": avg_metrics,
    "Best Fold Performance": best_row[metric_cols],
    "Worst Fold Performance": worst_row[metric_cols]
}).T

# Print execution time
end_time = time.time()
print(f"Execution time: {end_time - start_time:.2f} seconds")


In [ ]:
# Save summary results
summary_df.to_excel("summary_dataframe.xlsx")

## Results analysis

This section analyzes the results of the model training and evaluation.

In [ ]:
# Model performance: averages across the 5 folds, plus best and worst fold (reloads saved CV results)
# Load K-Fold results
results_df = pd.read_excel("kfold_performance_metrics_bert.xlsx")

# Compute and print average metrics
metric_cols = ["eval_loss", "eval_precision", "eval_recall", "eval_f1", "eval_accuracy", "cohen_kappa"]
avg_metrics = results_df[metric_cols].mean()
best_row = results_df.sort_values("eval_loss").iloc[0]
worst_row = results_df.sort_values("eval_loss", ascending=False).iloc[0]

print("\n Average Metrics Across Folds:")
print(avg_metrics)

print("\n Best Fold Performance:")
print(best_row)

print("\n Worst Fold Performance:")
print(worst_row)


# Save to file
summary1_df = pd.DataFrame({
    "Average Metrics": avg_metrics,
    "Best Fold Performance": best_row[metric_cols],
    "Worst Fold Performance": worst_row[metric_cols]
}).T
summary1_df.to_excel("kfold_summary1_metrics.xlsx")


In [ ]:
import seaborn as sns


## Visualizations
This section provides additional visualizations to better understand the model performance.

In [ ]:
# Visualize per-label metrics
try:
    per_variable_df = pd.read_excel("per_construct_cv_metrics.xlsx")

    # Sort by F1 score
    sorted_metrics = per_variable_df.sort_values(by="f1", ascending=False)

    # Plot F1 scores for each label
    plt.figure(figsize=(12, 8))
    bars = plt.barh(sorted_metrics["label"], sorted_metrics["f1"], color='skyblue')

    # Add value labels to the bars
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 0.01, bar.get_y() + bar.get_height()/2, f'{width:.2f}',
                 ha='left', va='center', fontweight='bold')

    plt.xlabel("F1 Score", fontsize=14)
    plt.ylabel("Label", fontsize=14)
    plt.title("F1 Score by Label", fontsize=16)
    plt.xlim(0, 1)
    plt.grid(axis="x", linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()

    # Display the metrics table
    print("Per-label performance metrics:")
    display(sorted_metrics)

    # Create a heatmap of precision, recall, and F1 for each label
    plt.figure(figsize=(14, 10))
    metrics_heatmap = sorted_metrics[['label', 'precision', 'recall', 'f1']].set_index('label')
    sns.heatmap(metrics_heatmap, annot=True, cmap='YlGnBu', fmt='.2f', linewidths=.5)
    plt.title('Precision, Recall, and F1 Score by Label', fontsize=16)
    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print("Per-construct CV metrics file not found. Run the model training first.")

In [ ]:
# Visualize tuned thresholds
try:
    with open("tuned_thresholds_per_fold.json", "r") as f:
        threshold_data = json.load(f)

    # Extract thresholds from all folds
    all_thresholds = {}
    for fold_data in threshold_data:
        for label, threshold in fold_data["thresholds"].items():
            if label not in all_thresholds:
                all_thresholds[label] = []
            all_thresholds[label].append(threshold)

    # Calculate average threshold for each label
    avg_thresholds = {label: np.mean(thresholds) for label, thresholds in all_thresholds.items()}
    std_thresholds = {label: np.std(thresholds) for label, thresholds in all_thresholds.items()}

    # Create a DataFrame for visualization
    threshold_df = pd.DataFrame({
        "label": list(avg_thresholds.keys()),
        "avg_threshold": list(avg_thresholds.values()),
        "std_threshold": list(std_thresholds.values())
    })

    # Sort by average threshold
    threshold_df = threshold_df.sort_values(by="avg_threshold")

    # Plot average thresholds
    plt.figure(figsize=(12, 8))
    bars = plt.barh(threshold_df["label"], threshold_df["avg_threshold"],
                    xerr=threshold_df["std_threshold"], capsize=5, alpha=0.7, color="skyblue")

    # Add value labels to the bars
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 0.01, bar.get_y() + bar.get_height()/2, f'{width:.2f}',
                 ha='left', va='center', fontweight='bold')

    plt.xlabel("Average Threshold", fontsize=14)
    plt.ylabel("Label", fontsize=14)
    plt.title("Optimized Classification Thresholds by Label", fontsize=16)
    plt.axvline(x=0.5, color="red", linestyle="--", label="Default threshold (0.5)")
    plt.grid(axis="x", linestyle="--", alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Display the threshold table
    print("Optimized classification thresholds by label:")
    display(threshold_df)
except FileNotFoundError:
    print("Threshold data file not found. Run the model training first.")

## Final training on the full training set

In [ ]:
# Final Training using Best Hyperparameters (on 100% of data)
import json

# Create dataset
final_dataset = TextDataset(
    texts=text_data,
    labels=labels,
    tokenizer=tokenizer
)

# Final training arguments
final_args = TrainingArguments(
    output_dir="./final_model",
    num_train_epochs=int(best_row["epochs"]),
    per_device_train_batch_size=int(best_row["batch_size"]),
    per_device_eval_batch_size=int(best_row["batch_size"] * 2),
    learning_rate=best_row["learning_rate"],
    weight_decay=best_row["weight_decay"],
    eval_strategy="no",  # No evaluation set
    logging_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=False,
    gradient_accumulation_steps=2,
    fp16=True,
    report_to="none"
)

# Initialize model
final_model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(topic_columns),
    problem_type="multi_label_classification"
)

# Create trainer
final_trainer = CustomTrainer(
    model=final_model,
    args=final_args,
    train_dataset=final_dataset,
    compute_metrics=None  # No evaluation metrics since no eval set
)

# Train and save
final_trainer.train()
final_model.save_pretrained("./final_model")
tokenizer.save_pretrained("./final_model")

print("\nFinal model trained on 100% of the training set and saved to ./final_model")

## Step 9 — Final model validation on held-out test data

In [ ]:
# Step 9 — Final model validation on the held-out test set (Section 3.9)
# Requires ./final_model (trained model + tokenizer) and optimal_thresholds.json from Step 8

# ========= CONFIGURATION =========
TEST_FILE = "/data/test_data.csv"  # user-supplied held-out test data
MODEL_PATH = "./final_model"
THRESHOLDS_FILE = "optimal_thresholds.json"

# ========= LOAD DATA =========
test_data = pd.read_csv(TEST_FILE)
test_data['pros'] = test_data['pros'].astype(str).fillna("")

# Assume first 2 columns are ID and 'pros', rest are labels.
# Reuse the training-time topic_columns (and its column order) rather than
# reassigning it from test_data: pred_logits below come from a model whose
# output order is fixed by the TRAINING label order, so scoring must use
# that same order or predictions get silently paired with the wrong construct.
test_topic_columns = test_data.columns[2:]
assert set(test_topic_columns) == set(topic_columns), (
    "test_data construct columns do not match train_data construct columns: "
    f"{set(test_topic_columns) ^ set(topic_columns)}"
)
labels_test = test_data[topic_columns].fillna(0).astype(int)  # reindexed to training order
text_data_test = test_data['pros'].tolist()

# ========= DATASET CLASS =========
class TextDataset(Dataset):
    def __init__(self, texts, labels=None, max_token_len=512):
        self.texts = texts
        self.labels = labels.reset_index(drop=True) if labels is not None else None
        self.max_token_len = max_token_len
        self.tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        inputs = self.tokenizer(
            self.texts[idx],
            add_special_tokens=True,
            max_length=self.max_token_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True
        )
        item = {
            'input_ids': torch.tensor(inputs['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(inputs['attention_mask'], dtype=torch.long)
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels.iloc[idx].values.astype(float), dtype=torch.float)
        return item

# ========= CUSTOM TRAINER =========
import torch.nn as nn
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.BCEWithLogitsLoss()
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ========= LOAD MODEL =========
model = BertForSequenceClassification.from_pretrained(MODEL_PATH)

# ========= INFERENCE =========
test_dataset = TextDataset(text_data_test, labels=labels_test)
trainer = CustomTrainer(
    model=model,
    args=TrainingArguments(output_dir="test_eval", per_device_eval_batch_size=8)
)
pred_logits = trainer.predict(test_dataset).predictions
pred_probs = torch.sigmoid(torch.tensor(pred_logits)).numpy()

# ========= APPLY THRESHOLDS =========
try:
    with open(THRESHOLDS_FILE, "r") as f:
        avg_thresholds = json.load(f)
        thresholds = [avg_thresholds[label] for label in topic_columns]
        pred_labels = (pred_probs >= thresholds).astype(int)
        print(f"\n Applied average tuned thresholds from '{THRESHOLDS_FILE}'")
except:
    pred_labels = (pred_probs >= 0.5).astype(int)
    print("\n  Thresholds file not found — using default threshold 0.5")




# ========= METRICS =========
y_true = labels_test.values
y_pred = pred_labels

metrics = {
    "precision_micro": precision_score(y_true, y_pred, average='micro'),
    "recall_micro": recall_score(y_true, y_pred, average='micro'),
    "f1_micro": f1_score(y_true, y_pred, average='micro'),
    "accuracy_micro": accuracy_score(y_true.flatten(), y_pred.flatten()),
    "cohen_kappa_micro": cohen_kappa_score(y_true.flatten(), y_pred.flatten())
}

# Per-label metrics
per_label_metrics = []
for i, label in enumerate(topic_columns):
    per_label_metrics.append({
        "label": label,
        "precision": precision_score(y_true[:, i], y_pred[:, i], zero_division=0),
        "recall": recall_score(y_true[:, i], y_pred[:, i], zero_division=0),
        "f1": f1_score(y_true[:, i], y_pred[:, i], zero_division=0),
        "accuracy": accuracy_score(y_true[:, i], y_pred[:, i]),
        "cohen_kappa": cohen_kappa_score(y_true[:, i], y_pred[:, i])
    })

# ========= EXPORT =========
pred_df = pd.DataFrame(y_pred, columns=topic_columns)
prob_df = pd.DataFrame(pred_probs, columns=[f"{c}_prob" for c in topic_columns])
output_df = pd.concat([test_data[['pros']].reset_index(drop=True), pred_df, prob_df], axis=1)

output_df.to_excel("test_predictions_with_probs.xlsx", index=False)
pd.DataFrame(per_label_metrics).to_excel("test_per_label_metrics.xlsx", index=False)
pd.DataFrame([metrics]).to_excel("test_overall_metrics.xlsx", index=False)

print("\n Evaluation complete.")
print(" Saved files:")
print(" - test_predictions_with_probs.xlsx")
print(" - test_per_label_metrics.xlsx")
print(" - test_overall_metrics.xlsx")



## Model inference (predict_labels)

In [ ]:
def predict_labels(text, model_path="./final_model"):
    """Predict labels for a given text using the trained model."""
    import os
    import json
    import torch
    import matplotlib.pyplot as plt
    from transformers import BertTokenizer, BertForSequenceClassification

    tokenizer = BertTokenizer.from_pretrained(model_path)
    model = BertForSequenceClassification.from_pretrained(model_path)

    # Load construct-specific thresholds (flat {construct: value} dict written in Step 8)
    THRESHOLDS_PATH = "optimal_thresholds.json"
    try:
        with open(THRESHOLDS_PATH, "r") as f:
            thresholds = {label: float(v) for label, v in json.load(f).items()}
    except (FileNotFoundError, json.JSONDecodeError):
        print(f"Warning: could not read {THRESHOLDS_PATH}. Using default threshold of 0.5.")
        thresholds = {label: 0.5 for label in topic_columns}

    # Tokenize input
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)

    # Get predictions
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probas = torch.sigmoid(logits).numpy()[0]

    # Apply thresholds - ensure they are floats
    results = {}
    for i, label in enumerate(topic_columns):
        threshold = float(thresholds.get(label, 0.5))  # Ensure it's a float
        probability = float(probas[i])
        predicted = probability >= threshold
        results[label] = {
            "probability": probability,
            "predicted": bool(predicted),
            "threshold": threshold
        }

    # Helper function to sort results
    def sort_results_by_probability(results):
        return {
            k: v for k, v in sorted(results.items(), key=lambda item: item[1]["probability"], reverse=True)
        }

    sorted_results = sort_results_by_probability(results)
    return sorted_results

# === Example usage ===
try:
    # Example: multi-construct assignment for a single review (generates Figure 4 in the paper)
    sample_text = "Great work environment with competitive pay and benefits. Supportive team and good work-life balance."

    predictions = predict_labels(sample_text)

    print("\n🔍 Sample prediction:")
    print(f"Text: {sample_text}\n")
    print("Predicted labels:")
    for label, info in predictions.items():
        if info["predicted"]:
            print(f"- {label}: {info['probability']:.4f} (threshold: {info['threshold']:.4f})")

    # Visualization
    import matplotlib.pyplot as plt

    labels_list = list(predictions.keys())
    probas = [predictions[label]["probability"] for label in labels_list]
    thresholds = [predictions[label]["threshold"] for label in labels_list]
    predicted = [predictions[label]["predicted"] for label in labels_list]
    colors = ['green' if pred else 'gray' for pred in predicted]

    plt.figure(figsize=(12, 8))
    bars = plt.barh(labels_list, probas, color=colors, alpha=0.7)

    for i, (label, threshold) in enumerate(zip(labels_list, thresholds)):
        plt.axvline(x=threshold, ymin=(i / len(labels_list)), ymax=((i + 1) / len(labels_list)),
                    color='red', linestyle='--', alpha=0.5)

    for bar, prob in zip(bars, probas):
        plt.text(prob + 0.01, bar.get_y() + bar.get_height()/2, f'{prob:.2f}',
                 va='center', fontweight='bold')

    plt.xlabel('Probability')
    plt.title('Prediction Probabilities')
    plt.xlim(0, 1.1)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig("sample1_prediction.png")
    plt.show()
    print("\n Sample prediction visualization saved to 'sample1_prediction.png'")

except Exception as e:
    print(f"\n️ Error during inference: {e}")



## Conclusion

This notebook implements a multi-label text classification model using BERT to categorize employee reviews based on various job aspects mentioned in the 'pros' section. The implementation includes:

1. **Hyperparameter Optimization**: Bayesian optimization (Optuna) over learning rate, batch size, weight decay, and number of epochs to find the optimal configuration.

2. **K-Fold Cross-Validation**: Uses 5-fold multi-label stratified cross-validation to ensure robust performance evaluation across different data splits.

3. **Dynamic Threshold Tuning**: Optimizes classification thresholds for each label individually to maximize F1 scores, rather than using a fixed threshold of 0.5.

4. **Comprehensive Evaluation**: Provides detailed performance metrics including precision, recall, F1 score, and Cohen's kappa, both overall and per label.

5. **Final Model Training**: Trains the final model on 100% of the training set using the selected hyperparameters.

6. **Inference**: Includes an inference function (`predict_labels`) for applying the trained model to new text.
